In [2]:
#!/usr/bin/env python3
"""
2D spatial interpolation using only top view (x,y) coordinates from electrode positions.
Similar to MNE's topoplot interpolation.
"""

import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
import pickle
import mne
from mne.channels import make_dig_montage


In [3]:
def load_avalanche_data(filepath, min_time_length=1, max_time_length=None):
    """
    Load avalanche data from pickle file
    
    Args:
        filepath: path to the pickle file
        min_time_length: minimum temporal length required (default: 15)
        max_time_length: maximum temporal length allowed (default: None, no upper limit)
    
    Returns:
        avalanche_arrays: list of avalanche arrays with temporal length in [min_time_length, max_time_length]
    """
    with open(filepath, 'rb') as f:
        avalanches = pickle.load(f)
    
    print(f"Loaded {len(avalanches)} participants")
    
    # Extract arrays from ALL participants  
    avalanche_arrays = []
    filtered_count_too_short = 0
    filtered_count_too_long = 0
    
    for participant_idx, participant_avalanches in enumerate(avalanches):
        for avalanche_idx, avalanche in enumerate(participant_avalanches):
            # avalanche is already a numpy array, not a list
            if hasattr(avalanche, 'shape') and avalanche.ndim == 2:
                temporal_length = avalanche.shape[1]
                
                # Check temporal length constraints
                if temporal_length < min_time_length:
                    filtered_count_too_short += 1
                elif max_time_length is not None and temporal_length > max_time_length:
                    filtered_count_too_long += 1
                else:
                    avalanche_arrays.append(avalanche)
                    if participant_idx == 0 and len(avalanche_arrays) <= 3:  # Debug first few
                        print(f"  Added avalanche {avalanche_idx}: shape {avalanche.shape}")
    
    print(f"Total avalanches extracted: {len(avalanche_arrays)}")
    print(f"Filtered out {filtered_count_too_short} avalanches with temporal length < {min_time_length}")
    if max_time_length is not None:
        print(f"Filtered out {filtered_count_too_long} avalanches with temporal length > {max_time_length}")
    
    return avalanche_arrays

In [4]:
def plot_improved_2d_mne(data, electrode_positions_file, indices=None):
    """
    Create improved 2D visualizations using MNE's best practices.
    
    Parameters:
    -----------
    data : list or array
        The avalanche data
    electrode_positions_file : str
        Path to the electrode positions JSON file
    indices : list, optional
        Specific indices to plot (e.g., [1, 2, 5, 10]). 
        If None, plots the first 3 avalanches (indices 0, 1, 2).
    """
    
    # Load electrode positions
    with open(electrode_positions_file, 'r') as f:
        electrode_data = json.load(f)
    
    channels = list(electrode_data.keys())
    
    # Create custom montage
    ch_pos = {}
    for ch in channels:
        pos = electrode_data[ch]
        ch_pos[ch] = [pos['x'], pos['y'], pos['z']]
    
    montage = make_dig_montage(ch_pos=ch_pos, coord_frame='head')
    
    # Set default indices if none provided
    if indices is None:
        indices = [0, 1, 2]  # Default to first 3 avalanches
    
    # Filter valid indices (within bounds of data)
    valid_indices = [idx for idx in indices if 0 <= idx < len(data)]
    
    # Warn if some indices are out of bounds
    invalid_indices = [idx for idx in indices if idx not in valid_indices]
    if invalid_indices:
        print(f"Warning: The following indices are out of bounds and will be skipped: {invalid_indices}")
        print(f"Data has {len(data)} avalanches (indices 0 to {len(data)-1})")
    
    for avalanche_idx in valid_indices:
        avalanche = data[avalanche_idx]
        array_data = np.array(avalanche)
        n_electrodes, n_timepoints = array_data.shape
        
        # Create MNE info
        info = mne.create_info(ch_names=channels[:n_electrodes], sfreq=250, ch_types='eeg')
        
        # Create evoked object
        evoked = mne.EvokedArray(array_data, info)
        evoked.set_montage(montage)
        
        # Adaptive vlim
        data_max = np.max(np.abs(array_data))
        vlim = (-data_max, data_max) if data_max > 0 else (-1, 1)
        
        # Create time points for visualization
        times = np.arange(n_timepoints) / 250.0
        
        # MNE's improved 2D visualization
        fig = evoked.plot_topomap(
            times=times,
            show=False,
            ncols=10,
            nrows='auto',
            size=2.5,
            vlim=vlim,
            units='Activity',
            image_interp='linear',
            scalings=1,
            cmap='RdBu_r',
            res=256,
            extrapolate='head',
            outlines='head',
            contours=0,
            sensors=True,
            sphere='auto'  # Let MNE auto-detect best head model
        )
        
        fig.suptitle(f'Avalanche {avalanche_idx} - MNE Enhanced Topomaps', fontsize=16)
        plt.show()


# Example usage:
# plot_improved_2d_mne(data, electrode_file)  # Plots indices [0, 1, 2] by default
# plot_improved_2d_mne(data, electrode_file, indices=[0, 5, 15, 20, 25])  # Another example

In [ ]:
avalanche_arrays = load_avalanche_data('/home/novicky/clean_code/processed_avalanches_250Hz/filtered_avalanche_peaks_250hz_twopoles.pkl', min_time_length=1, max_time_length=None)

indices = [36, 45, 99, 58, 6, 3, 49, 42, 21, 48, 17, 44, 10, 33, 61]
plot_improved_2d_mne(avalanche_arrays, '/home/novicky/clean_code/electrode_positions.json', indices=indices)  # Plots specific indices